In [1]:
from sqlalchemy import create_engine, text
import bcrypt
import dotenv
import os

In [2]:
dotenv.load_dotenv("db_palete_caixa.env", override=True)

True

In [3]:
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")
USER = os.getenv("USER")
DATABASE = os.getenv("DATABASE")
PASS = os.getenv("PASS")
print(HOST, PORT, USER, DATABASE)

localhost 5432 postgres db_palete_caixa


In [4]:
if PASS:
    print(f"{'\u2022' * 5}")
else:
    print("Senha não encontrada")

•••••


In [5]:
url = f"postgresql+psycopg2://{USER}:{PASS}@{HOST}:{PORT}/{DATABASE}"

In [6]:
engine = create_engine(
    url, connect_args={"sslmode": "prefer"}
)

In [7]:
import pandas as pd
df_tabelas = pd.read_sql("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_name IN ('clientes', 'tipos_status', 'tipos_usuarios')
        ORDER BY table_name;
    """, engine)
df_tabelas

,table_name
0,clientes
1,tipos_status
2,tipos_usuarios


In [8]:
cria_tabelas = """
CREATE TABLE IF NOT EXISTS tipos_status (
    id SERIAL PRIMARY KEY,
    nome_status VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS tipos_usuarios (
    id SERIAL PRIMARY KEY,
    tipo VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS clientes (
    id                  SERIAL PRIMARY KEY,
    cnpj                VARCHAR(14) NOT NULL UNIQUE,
    razao_social        VARCHAR(255) NOT NULL,
    nome_fantasia       VARCHAR(255),
    nome_contato      VARCHAR(255),
    telefone_contato  VARCHAR(20),
    email_contato     VARCHAR(255),
    nome_usuario      VARCHAR(50)  NOT NULL UNIQUE,
    senha_hash        VARCHAR(255) NOT NULL,
    status            VARCHAR(30)  NOT NULL DEFAULT 'ativo',
    tipo              VARCHAR(30)  NOT NULL DEFAULT 'regular',
    data_criacao      TIMESTAMP    NOT NULL DEFAULT NOW(),
    deletado          BOOLEAN      NOT NULL DEFAULT FALSE,
    deletado_em       TIMESTAMP
);
"""

with engine.begin() as conn:
    conn.execute(text(cria_tabelas))
print("Tabelas criadas com sucesso!!!")

Tabelas criadas com sucesso!!!


In [9]:
inserir_dados = """
INSERT INTO tipos_status (nome_status)
VALUES ('ativo'), ('inativo'), ('suspenso')
ON CONFLICT (nome_status) DO NOTHING;

INSERT INTO tipos_usuarios (tipo)
VALUES ('admin'), ('regular')
ON CONFLICT (tipo) DO NOTHING;
"""

with engine.begin() as conn:
    conn.execute(text(inserir_dados))
print("Dados inseridos com sucesso")

Dados inseridos com sucesso


In [10]:
df = pd.read_sql(sql="SELECT * FROM tipos_usuarios",
                 con=engine)
df

,id,tipo
0,1,admin
1,2,regular


In [11]:
# # tabelas: tipos_status, tipos_usarios, clientes
drop_tabelas = """
DROP TABLE IF EXISTS tabela_inexistente;
--DROP TABLE IF EXISTS tipos_status;
--DROP TABLE IF EXISTS tipos_usuarios;
--DROP TABLE IF EXISTS clientes;
"""

# truncate_tabelas = """
# --TRUNCATE TABLE tipos_status;
# --TRUNCATE TABLE tipos_usuarios;
# --TRUNCATE TABLE clientes;
# """

with engine.begin() as conn:
    conn.execute(text(drop_tabelas))
print("Tabela(s) excluidas com sucesso")

Tabela(s) excluidas com sucesso


In [12]:
adicionar_coluna = """
ALTER TABLE tipos_status
ADD COLUMN IF NOT EXISTS descricao VARCHAR(255)
"""

with engine.begin() as conn:
    conn.execute(text(adicionar_coluna))

df_tipos_status = pd.read_sql(sql="SELECT * FROM tipos_status",
                              con=engine)
df_tipos_status

,id,nome_status,descricao
0,1,ativo,NaN
1,2,inativo,NaN
2,3,suspenso,Usuário com licença fora do prazo


In [13]:
update_tabela = """
    UPDATE tipos_status
    SET descricao = 'Usuário com licença fora do prazo'
    WHERE nome_status = 'suspenso';
"""

with engine.begin() as conn:
    conn.execute(text(update_tabela))

df_tipos_status = pd.read_sql("SELECT * FROM tipos_status",
                              engine)
df_tipos_status

,id,nome_status,descricao
0,1,ativo,NaN
1,2,inativo,NaN
2,3,suspenso,Usuário com licença fora do prazo


In [14]:
cnpj         = "12345678000190"
razao_social = "Prandiano"
nome_contato = "Ricieri"
nome_usuario = "prandiano"
tipo         = "admin"

In [ ]:
import getpass
senha = getpass.getpass("Insira sua senha: ")
confirmar = getpass.getpass("Confirme sua senha: ")
if senha == confirmar:
    senha_hash = bcrypt.hashpw(senha.encode(), 
                           bcrypt.gensalt()).decode()
    print("senhas coincidem")
else:
    print("Senhas não coincidem")

print(senha_hash)